# HTML Schedule Export
This notebook parametrises the HTML/Jinja pipeline to generate branded schedules for classes, teachers (_pros_), and rooms (_locals_).

- Use the `RENDERER` parameter below to pick `wkhtmltopdf` (good default) or `chromium` (requires a local Chrome/Chromium binary) when you want Chrome-quality PDFs.
- The notebook also writes responsive HTML pages alongside PDFs so you can open them in a browser and print at 100% if needed.


In [ ]:
from pathlib import Path

# --- Pipeline parameters ---
INPUT_FILE = Path("../input/inputSM_2025-2026_3_10.xlsx")
SHEET_NAME = "input"
SEMVER = "2025-2026-v0.9"
TEMPLATE_ROOT = Path("../templates/html")
TEMPLATE_NAME = "schedule.html.j2"
OUTPUT_DIR = Path("../output/html")
INCLUDE_GENERAL = True  # Include classe entière rows when rendering subgroup sheets
RENDERER = "chromium"  # pdf renderer: weasyprint, wkhtmltopdf, or chromium


In [ ]:
import shutil
from shutil import which
print("wkhtmltopdf:", which("wkhtmltopdf"))


In [ ]:
import pandas as pd

df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)
display(df.head())

In [ ]:
from fragenda.html_export import generate_class_pdfs, generate_scope_pdfs

OUTPUT_DIR_CLASSES = OUTPUT_DIR / "classes"
OUTPUT_DIR_PROS = OUTPUT_DIR / "pros"
OUTPUT_DIR_LOCALS = OUTPUT_DIR / "locals"


## Classes

In [ ]:
class_pdfs = generate_class_pdfs(
    dataframe=df,
    semver=SEMVER,
    template_dir=TEMPLATE_ROOT,
    template_name=TEMPLATE_NAME,
    output_dir=OUTPUT_DIR_CLASSES,
    include_general=INCLUDE_GENERAL,
    renderer=RENDERER,
)
print(f"Generated {len(class_pdfs)} class schedule PDFs → {OUTPUT_DIR_CLASSES.resolve()}")

## Pros (Teachers)

In [ ]:
pro_pdfs = generate_scope_pdfs(
    dataframe=df,
    scope="pro",
    semver=SEMVER,
    template_dir=TEMPLATE_ROOT,
    template_name=TEMPLATE_NAME,
    output_dir=OUTPUT_DIR_PROS,
    renderer=RENDERER,
)
print(f"Generated {len(pro_pdfs)} teacher schedule PDFs → {OUTPUT_DIR_PROS.resolve()}")

## Locals (Rooms)

In [ ]:
local_pdfs = generate_scope_pdfs(
    dataframe=df,
    scope="local",
    semver=SEMVER,
    template_dir=TEMPLATE_ROOT,
    template_name=TEMPLATE_NAME,
    output_dir=OUTPUT_DIR_LOCALS,
    renderer=RENDERER,
)
print(f"Generated {len(local_pdfs)} room schedule PDFs → {OUTPUT_DIR_LOCALS.resolve()}")

## Next steps
- Adjust the parameters cell to target a different export, version tag, or template variant.
- Review the PDFs inside the `output/html/` subdirectories and share them.
- Keep the original Excel-based flow for legacy consumers.

In [ ]:
import shutil
from pathlib import Path
from pypdf import PdfWriter
from tqdm.auto import tqdm

release_tag = SEMVER.split('-')[-1]
release_root = (OUTPUT_DIR / f"horaires_{release_tag}").resolve()
if release_root.exists():
    shutil.rmtree(release_root)
release_root.mkdir(parents=True, exist_ok=True)

scopes = [
    (class_pdfs, 'classes'),
    (pro_pdfs, 'pros'),
    (local_pdfs, 'locaux'),
]

for pdf_list, scope in scopes:
    if not pdf_list:
        print(f"Skipping {scope}, no PDFs")
        continue
    folder_name = f"horaire_{scope}-{SEMVER}"
    dest_dir = release_root / folder_name
    dest_dir.mkdir(parents=True, exist_ok=True)

    writer = PdfWriter()
    print(f"↳ Packaging {scope} ({len(pdf_list)} files)")
    for pdf_path in tqdm(sorted(pdf_list), unit="pdf"):
        pdf_path = Path(pdf_path)
        shutil.copy2(pdf_path, dest_dir / pdf_path.name)
        writer.append(str(pdf_path))
    combined_pdf = release_root / f"{folder_name}.pdf"
    with combined_pdf.open('wb') as fh:
        writer.write(fh)

zip_target = release_root.with_name(f"horaires_{release_tag}.zip")
if zip_target.exists():
    zip_target.unlink()
shutil.make_archive(str(zip_target.with_suffix("")), 'zip', release_root)

print(f"Release folder → {release_root}")
print(f"Archive created → {zip_target}")
